# Un bucle de entrenamiento escrito a mano: `expval` y `jacobian`

En [`07a`](07a_clasificador_nativo.ipynb) y [`07b`](07b_lecturas_y_perdidas.ipynb), `fit` hace todo el entrenamiento: ejecuta los circuitos, calcula la pérdida y mueve los parámetros. A veces la pérdida o el bucle tienen que escribirse fuera de `fit`: una pérdida que `fit` no tiene, un modelo con partes clásicas alrededor, u otra biblioteca de entrenamiento. Para eso `polypus.qml` tiene dos primitivas que no calculan ninguna pérdida:

- `qml.expval(model, x, params)` devuelve el valor esperado de cada observable de la lectura para cada fila de `x`.
- `qml.jacobian(model, x, params)` devuelve sus derivadas respecto a los parámetros, calculadas de forma exacta con la regla de desplazamiento de parámetros, parameter-shift.

Con ellas, este notebook escribe a mano un descenso de gradiente.

## Datos y modelo

Hay seis muestras con dos características cada una: las tres primeras en torno a ángulos pequeños y objetivo `+1` y las tres últimas en torno a ángulos grandes y objetivo `−1`. El modelo es el de [`07a`](07a_clasificador_nativo.ipynb), con una lectura de `⟨Z₀⟩` sin decisión, así que su salida es el propio valor esperado. Los parámetros son ahora un array normal de NumPy, aquí un punto de partida aleatorio:

In [ ]:
import numpy as np
from polypus import qml

X = np.array([[0.2, 0.1], [0.4, 0.3], [0.1, 0.5], [2.7, 2.9], [3.0, 2.6], [2.8, 3.1]])
Y = np.array([1.0, 1.0, 1.0, -1.0, -1.0, -1.0])

modelo = qml.Model(
    qml.Encoder(2).angle_encoder("ry"),
    qml.Ansatz(2).real_amplitudes(1),
    qml.Readout([qml.Z(0)]),
)
params = np.random.default_rng(7).uniform(-np.pi, np.pi, modelo.num_params)
print(modelo.num_params, "parámetros,", modelo.num_observables, "observable")

## `expval`: las salidas del modelo

`expval` devuelve un array con una fila por muestra y una columna por observable, aquí 6 × 1. Admite los mismos argumentos de ejecución que `TrainedModel.run`: `exact=True` calcula los valores esperados exactos, mientras que con shots cada valor lleva ruido de muestreo:

In [ ]:
valores = qml.expval(modelo, X, params, exact=True)
print(valores.shape)
print(np.round(valores[:, 0], 3))
print(np.round(qml.expval(modelo, X, params, shots=1000, seed=3)[:, 0], 3))

## `jacobian`: cómo cambian las salidas con cada parámetro

`jacobian` añade una dimensión, la de los parámetros: la entrada `[i, k, j]` es cuánto cambia el observable `k` de la fila `i` cuando se mueve el parámetro `j`. La regla parameter-shift obtiene cada entrada con dos circuitos por parámetro, con su ángulo desplazado hacia arriba y hacia abajo, y es exacta, no una aproximación. Una diferencia finita con un paso muy pequeño, calculada con `expval`, da los mismos números para el parámetro 0:

In [ ]:
por_parametros = qml.jacobian(modelo, X, params, exact=True)
print(por_parametros.shape)

h = 1e-4
desplazamiento = np.zeros_like(params)
desplazamiento[0] = h
diferencia_finita = (
    qml.expval(modelo, X, params + desplazamiento, exact=True)
    - qml.expval(modelo, X, params - desplazamiento, exact=True)
) / (2 * h)
print(np.round(por_parametros[:, 0, 0], 6))
print(np.round(diferencia_finita[:, 0], 6))

## El bucle de entrenamiento

La pérdida es un error cuadrático, `mean((⟨Z₀⟩ − y)²)`. Su gradiente sale de la regla de la cadena: `2·mean((⟨Z₀⟩ − y)·∂⟨Z₀⟩/∂θ)`, que necesita justo los dos arrays de arriba. Cada paso calcula los dos y mueve los parámetros en contra del gradiente, con una tasa de aprendizaje de 0.3:

In [ ]:
import matplotlib.pyplot as plt

perdidas = []
for paso in range(40):
    valores = qml.expval(modelo, X, params, exact=True)
    perdidas.append(float(np.mean((valores[:, 0] - Y) ** 2)))
    por_parametros = qml.jacobian(modelo, X, params, exact=True)
    # regla de la cadena: d loss / d theta = 2 * mean((<Z0> - y) * d<Z0>/d theta)
    gradiente = 2 * (valores[:, 0] - Y) @ por_parametros[:, 0, :] / len(Y)
    params = params - 0.3 * gradiente

print("pérdida:", round(perdidas[0], 4), "->", round(perdidas[-1], 4))
print(np.round(qml.expval(modelo, X, params, exact=True)[:, 0], 3))

plt.figure(figsize=(5, 3))
plt.plot(perdidas)
plt.xlabel("paso")
plt.ylabel("error cuadrático")
plt.show()

Tras 40 pasos los valores esperados quedan cerca de los objetivos, `+1` para las tres primeras muestras y `−1` para las tres últimas.

## Derivadas respecto a las entradas

`wrt="inputs"` deriva respecto a las características de cada muestra en vez de respecto a los parámetros, solo con encoders de ángulo e IQP. Muestra lo sensible que es el modelo entrenado a cada característica, y es lo que permite que también aprenda una capa clásica colocada antes del modelo, como en el siguiente notebook. `wrt="both"` devuelve los dos jacobianos a la vez:

In [ ]:
por_entradas = qml.jacobian(modelo, X, params, wrt="inputs", exact=True)
print(por_entradas.shape)
print(np.round(por_entradas[:, 0, :], 3))

## Resumen

Este notebook ha calculado las salidas de un modelo nativo con `qml.expval` y sus derivadas exactas con `qml.jacobian`, las ha comparado con una diferencia finita, ha entrenado el modelo con un descenso de gradiente propio sobre un error cuadrático, y lo ha derivado respecto a sus entradas con `wrt="inputs"`.

El siguiente notebook usa estas mismas dos primitivas dentro de PyTorch, para meter un modelo nativo en una red híbrida cuántico-clásica.

## Siguiente paso

Continuar con: [`07d_hibrida_pytorch.ipynb`](07d_hibrida_pytorch.ipynb).